# Position-coded nodal RSAM heatmaps

This notebook reads the RSAM archive recomputed from the position-coded SDS
archive. The station code is the along-line position in centimetres:

```text
station code 15010 -> 150.10 m
```

No serial-number mapping CSV or deployment-position lookup is required.
Because repeat surveys can encode the same nominal receiver position a few
centimetres differently, positions within a configurable tolerance are
clustered before plotting. Measurements from different deployment location
codes in the same position cluster are then merged by time and position.

The notebook generates comparable heatmaps for:

- T1 DPZ, DPN, and DPE
- T3 DPZ, DPN, and DPE

All plots use one shared color scale calculated from the pooled finite values.

In [1]:
from __future__ import annotations

from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from obspy import UTCDateTime

from flovopy.processing.sam import RSAM

## Configuration

In [2]:
SAM_DIR = Path(
    "/Volumes/tachyon/LBSSP_DATA/nodal_rsam_position_codes"
)
OUTPUT_DIR = SAM_DIR / "position_time_heatmaps"

RUNS = {
    "T1": {
        "start": UTCDateTime("2026-05-16T00:00:00"),
        "end": UTCDateTime("2026-05-21T00:00:00"),
    },
    "T3": {
        "start": UTCDateTime("2026-05-19T00:00:00"),
        "end": UTCDateTime("2026-05-20T00:00:00"),
    },
}

CHANNELS = {
    "Z": "DPZ",
    "N": "DPN",
    "E": "DPE",
}

SAMPLING_INTERVAL_S = 60
RSAM_EXTENSION = "csv"
METRIC = "median"
CADENCE = "1min"

# Position codes from repeat deployments that differ by no more than this
# amount are treated as the same physical receiver location.
POSITION_TOLERANCE_M = 0.25

# Each merged station position is shown as a fixed-height band.
NODE_HEIGHT_M = 2.0
Y_RESOLUTION_M = 1.0

USE_LOG10 = True
GLOBAL_PERCENTILES = (2.0, 98.0)

FIGSIZE = (13, 7)
DPI = 200
VERBOSE_RSAM_READ = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"RSAM root:  {SAM_DIR}")
print(f"Output dir: {OUTPUT_DIR}")

RSAM root:  /Volumes/tachyon/LBSSP_DATA/nodal_rsam_position_codes
Output dir: /Volumes/tachyon/LBSSP_DATA/nodal_rsam_position_codes/position_time_heatmaps


## Data preparation functions

In [3]:
def parse_seed_id(seed_id: str) -> tuple[str, str, str, str]:
    """Split a SEED identifier into network, station, location, channel."""
    parts = seed_id.split(".")
    if len(parts) != 4:
        raise ValueError(
            f"Expected NET.STA.LOC.CHA, got {seed_id!r}"
        )
    return tuple(parts)


def station_code_to_position_m(station: str) -> float:
    """Convert a position-coded station name in centimetres to metres."""
    text = str(station).strip()

    if text.endswith(".0"):
        text = text[:-2]

    if not text.isdigit():
        raise ValueError(
            f"Station code {station!r} is not an integer centimetre position."
        )

    return int(text) / 100.0


def dataframe_time_series(
    dataframe: pd.DataFrame,
    metric: str,
) -> pd.Series:
    """Extract a UTC-indexed numeric RSAM metric series."""
    if metric not in dataframe.columns:
        raise KeyError(
            f"Metric {metric!r} not present; "
            f"available columns: {list(dataframe.columns)}"
        )

    if isinstance(dataframe.index, pd.DatetimeIndex):
        times = pd.to_datetime(dataframe.index, utc=True)
    else:
        time_column = next(
            (
                column
                for column in (
                    "time",
                    "datetime",
                    "date",
                    "timestamp",
                )
                if column in dataframe.columns
            ),
            None,
        )

        if time_column is None:
            numeric_index = pd.to_numeric(
                pd.Index(dataframe.index),
                errors="coerce",
            )

            if np.isfinite(numeric_index).all():
                times = pd.to_datetime(
                    numeric_index,
                    unit="s",
                    utc=True,
                )
            else:
                times = pd.to_datetime(
                    dataframe.index,
                    utc=True,
                    errors="coerce",
                )
        else:
            values = dataframe[time_column]
            times = pd.to_datetime(
                values,
                unit=(
                    "s"
                    if pd.api.types.is_numeric_dtype(values)
                    else None
                ),
                utc=True,
                errors="coerce",
            )

    values = pd.to_numeric(
        dataframe[metric],
        errors="coerce",
    ).to_numpy()

    series = pd.Series(values, index=times)
    return series[~series.index.isna()].sort_index()


def cluster_positions(
    positions: pd.Series,
    tolerance_m: float,
) -> tuple[pd.Series, pd.DataFrame]:
    """
    Merge nearby position codes that represent the same physical receiver.

    A cluster is built in sorted order and is allowed a total span no greater
    than ``tolerance_m``. Every member is replaced by the cluster mean.
    """
    if tolerance_m < 0:
        raise ValueError("tolerance_m cannot be negative.")

    unique_positions = np.sort(
        positions.dropna().unique().astype(float)
    )

    if len(unique_positions) == 0:
        return positions.copy(), pd.DataFrame()

    clusters: list[list[float]] = []
    current = [float(unique_positions[0])]

    for position in unique_positions[1:]:
        position = float(position)

        if position - current[0] <= tolerance_m + 1e-12:
            current.append(position)
        else:
            clusters.append(current)
            current = [position]

    clusters.append(current)

    position_lookup: dict[float, float] = {}
    summary_rows: list[dict[str, object]] = []

    for cluster_number, members in enumerate(clusters, start=1):
        mean_position = float(np.mean(members))

        for member in members:
            position_lookup[member] = mean_position

        summary_rows.append(
            {
                "cluster": cluster_number,
                "mean_position_m": mean_position,
                "minimum_position_m": min(members),
                "maximum_position_m": max(members),
                "span_m": max(members) - min(members),
                "original_position_count": len(members),
                "original_positions_m": ", ".join(
                    f"{member:.2f}" for member in members
                ),
            }
        )

    clustered = positions.map(
        lambda value: (
            position_lookup[float(value)]
            if pd.notna(value)
            else np.nan
        )
    )

    return clustered, pd.DataFrame(summary_rows)


def build_long_table(
    rsam: RSAM,
    *,
    metric: str,
    channel: str,
    position_tolerance_m: float,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    """Build a long table for one DP channel and merge nearby positions."""
    rows: list[pd.DataFrame] = []
    skipped: list[tuple[str, str]] = []
    matched_seed_ids: list[str] = []

    for seed_id, dataframe in rsam.dataframes.items():
        try:
            network, station, location, seed_channel = (
                parse_seed_id(seed_id)
            )

            if seed_channel != channel:
                continue

            original_position_m = station_code_to_position_m(
                station
            )
            series = dataframe_time_series(dataframe, metric)

        except Exception as exc:
            skipped.append((seed_id, str(exc)))
            continue

        values = series.to_numpy(dtype=float)
        good = np.isfinite(values)

        if not good.any():
            continue

        matched_seed_ids.append(seed_id)

        rows.append(
            pd.DataFrame(
                {
                    "time": series.index[good],
                    "value": values[good],
                    "network": network,
                    "station": station,
                    "location": location,
                    "channel": seed_channel,
                    "x_m_original": original_position_m,
                    "seed_id": seed_id,
                }
            )
        )

    if skipped:
        print(f"Skipped {len(skipped)} trace IDs. First examples:")
        for seed_id, reason in skipped[:15]:
            print(f"  {seed_id}: {reason}")

    if not rows:
        raise RuntimeError(
            f"No finite RSAM samples found for channel {channel!r}."
        )

    table = pd.concat(rows, ignore_index=True)

    table["x_m"], position_clusters = cluster_positions(
        table["x_m_original"],
        tolerance_m=position_tolerance_m,
    )

    return table, position_clusters, sorted(matched_seed_ids)

## Grid and plotting functions

Rows from different deployment location codes are merged when they have the
same position-coded station name. If more than one value occurs at the same
position and minute, the median is used.

In [4]:
def make_grid(
    table: pd.DataFrame,
    cadence: str,
) -> tuple[pd.DatetimeIndex, np.ndarray, np.ndarray]:
    """Create one resampled series per physical node position."""
    positions = np.sort(
        table["x_m"].dropna().unique().astype(float)
    )

    t0 = table["time"].min().floor(cadence)
    t1 = table["time"].max().ceil(cadence)

    times = pd.date_range(
        t0,
        t1,
        freq=cadence,
        tz="UTC",
    )

    grid = np.full(
        (len(positions), len(times)),
        np.nan,
    )

    for row_index, position_m in enumerate(positions):
        subset = table.loc[
            table["x_m"] == position_m,
            ["time", "value"],
        ]

        series = (
            subset
            .set_index("time")["value"]
            .sort_index()
            .resample(cadence)
            .median()
            .reindex(times)
        )

        grid[row_index, :] = series.to_numpy(dtype=float)

    return times, positions, grid


def make_fixed_height_position_grid(
    positions: np.ndarray,
    grid: np.ndarray,
    *,
    node_height_m: float,
    y_resolution_m: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Expand each node series into a fixed-height vertical band."""
    if node_height_m <= 0:
        raise ValueError("node_height_m must be positive.")

    if y_resolution_m <= 0:
        raise ValueError("y_resolution_m must be positive.")

    # Display at the nearest whole metre, as requested for the report figure.
    display_positions = np.rint(positions).astype(int)

    duplicate_counts = (
        pd.Series(display_positions)
        .value_counts()
        .loc[lambda values: values > 1]
    )

    if not duplicate_counts.empty:
        details = []

        for rounded_position in duplicate_counts.index:
            original_positions = positions[
                display_positions == rounded_position
            ]
            details.append(
                f"{rounded_position} m <- "
                + ", ".join(
                    f"{position:.2f} m"
                    for position in original_positions
                )
            )

        raise ValueError(
            "Rounding position-coded stations to whole metres merged "
            "distinct nodes:\n"
            + "\n".join(details)
        )

    half_height = node_height_m / 2.0

    y_min = np.floor(
        display_positions.min() - half_height
    )
    y_max = np.ceil(
        display_positions.max() + half_height
    )

    y_edges = np.arange(
        y_min,
        y_max + y_resolution_m,
        y_resolution_m,
        dtype=float,
    )

    y_lower = y_edges[:-1]
    y_upper = y_edges[1:]

    expanded_grid = np.full(
        (len(y_edges) - 1, grid.shape[1]),
        np.nan,
        dtype=float,
    )

    for source_row, display_position in enumerate(
        display_positions
    ):
        band_min = display_position - half_height
        band_max = display_position + half_height

        target_rows = (
            (y_lower >= band_min - 1e-9)
            & (y_upper <= band_max + 1e-9)
        )

        expanded_grid[target_rows, :] = grid[source_row, :]

    return display_positions, y_edges, expanded_grid


def datetime_bin_edges(
    times: pd.DatetimeIndex,
    cadence: str,
) -> np.ndarray:
    """Create explicit left-aligned pcolormesh time-bin edges."""
    if len(times) == 0:
        raise ValueError("No times supplied.")

    delta = pd.to_timedelta(cadence)

    edges = times.append(
        pd.DatetimeIndex([times[-1] + delta])
    )

    return mdates.date2num(edges.to_pydatetime())


def prepare_plot_data(
    table: pd.DataFrame,
    *,
    cadence: str,
    node_height_m: float,
    y_resolution_m: float,
    use_log10: bool,
) -> dict[str, object]:
    times, physical_positions, node_grid = make_grid(
        table,
        cadence,
    )

    display_positions, y_edges, display_grid = (
        make_fixed_height_position_grid(
            physical_positions,
            node_grid,
            node_height_m=node_height_m,
            y_resolution_m=y_resolution_m,
        )
    )

    if use_log10:
        with np.errstate(
            divide="ignore",
            invalid="ignore",
        ):
            z = np.log10(display_grid)
    else:
        z = display_grid.copy()

    z[~np.isfinite(z)] = np.nan

    if not np.isfinite(z).any():
        raise RuntimeError(
            "No finite transformed values are available for plotting."
        )

    return {
        "times": times,
        "physical_positions": physical_positions,
        "display_positions": display_positions,
        "y_edges": y_edges,
        "z": z,
    }


def plot_heatmap(
    plot_data: dict[str, object],
    *,
    outfile: Path,
    title: str,
    metric: str,
    channel: str,
    cadence: str,
    use_log10: bool,
    vmin: float,
    vmax: float,
    figsize: tuple[float, float],
    dpi: int,
) -> tuple[plt.Figure, plt.Axes]:
    times = plot_data["times"]
    display_positions = plot_data["display_positions"]
    y_edges = plot_data["y_edges"]
    z = plot_data["z"]

    x_edges = datetime_bin_edges(times, cadence)

    fig, ax = plt.subplots(
        figsize=figsize,
        constrained_layout=True,
    )

    mesh = ax.pcolormesh(
        x_edges,
        y_edges,
        z,
        shading="flat",
        vmin=vmin,
        vmax=vmax,
    )

    ax.set_xlabel("Time (UTC)")
    ax.set_ylabel("Position along profile (m)")
    ax.set_ylim(y_edges[0], y_edges[-1])

    locator = mdates.AutoDateLocator(
        minticks=5,
        maxticks=9,
    )
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(
        mdates.ConciseDateFormatter(locator)
    )

    if len(display_positions) <= 50:
        ax.set_yticks(display_positions)

    ax.grid(False)
    ax.set_title(title)

    colorbar = fig.colorbar(
        mesh,
        ax=ax,
        pad=0.015,
    )
    colorbar.set_label(
        (
            f"log10({metric} RSAM), {channel}"
            if use_log10
            else f"{metric} RSAM, {channel}"
        )
    )

    outfile.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    fig.savefig(
        outfile,
        dpi=dpi,
        bbox_inches="tight",
    )

    print(f"Wrote {outfile}")

    return fig, ax

## Load each network once

In [5]:
rsam_by_network: dict[str, RSAM] = {}

for network, time_window in RUNS.items():
    print(
        f"Reading {network}: "
        f"{time_window['start']} to {time_window['end']}"
    )

    rsam = RSAM.read(
        time_window["start"],
        time_window["end"],
        SAM_DIR=str(SAM_DIR),
        network=network,
        sampling_interval=SAMPLING_INTERVAL_S,
        ext=RSAM_EXTENSION,
        verbose=VERBOSE_RSAM_READ,
    )

    rsam_by_network[network] = rsam

    print(
        f"  Loaded {len(rsam.dataframes)} RSAM dataframes."
    )

Reading T1: 2026-05-16T00:00:00.000000Z to 2026-05-21T00:00:00.000000Z
Dataframe with 1376 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 1376 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 1376 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 70 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 70 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 70 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 1376 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 1376 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 1376 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 69 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 69 rows is already on a regular 60 s grid based on column 'time'
Dataframe with 69 rows is already on 

## Build the six network/component datasets

In [6]:
results: dict[tuple[str, str], dict[str, object]] = {}

for network, rsam in rsam_by_network.items():
    for component, channel in CHANNELS.items():
        key = (network, component)

        print(
            f"\nPreparing {network} {channel}"
        )

        try:
            table, position_clusters, matched_seed_ids = build_long_table(
                rsam,
                metric=METRIC,
                channel=channel,
                position_tolerance_m=POSITION_TOLERANCE_M,
            )

            plot_data = prepare_plot_data(
                table,
                cadence=CADENCE,
                node_height_m=NODE_HEIGHT_M,
                y_resolution_m=Y_RESOLUTION_M,
                use_log10=USE_LOG10,
            )

        except RuntimeError as exc:
            print(f"  Skipping {network} {channel}: {exc}")
            continue

        print(
            f"  Matched {len(matched_seed_ids)} SEED IDs"
        )
        print(
            f"  Original position codes: "
            f"{table['x_m_original'].nunique()}"
        )
        print(
            f"  Merged physical positions: "
            f"{table['x_m'].nunique()}"
        )
        print(
            f"  Location codes: "
            f"{sorted(table['location'].unique())}"
        )

        results[key] = {
            "channel": channel,
            "table": table,
            "position_clusters": position_clusters,
            "matched_seed_ids": matched_seed_ids,
            "plot_data": plot_data,
        }

if not results:
    raise RuntimeError(
        "None of the requested RSAM datasets could be prepared."
    )

print(f"\nPrepared {len(results)} plot datasets.")


Preparing T1 DPZ


ValueError: Rounding position-coded stations to whole metres merged distinct nodes:
104 m <- 103.98 m, 104.08 m
124 m <- 124.02 m, 124.13 m
132 m <- 132.02 m, 132.16 m
120 m <- 120.05 m, 120.18 m
116 m <- 116.00 m, 116.11 m
136 m <- 136.05 m, 136.20 m
112 m <- 112.02 m, 112.15 m
108 m <- 108.00 m, 108.14 m
128 m <- 128.01 m, 128.15 m

## Calculate shared color limits

The same `vmin` and `vmax` are calculated from all successfully prepared plots.

In [ ]:
pooled_values = np.concatenate(
    [
        result["plot_data"]["z"][
            np.isfinite(result["plot_data"]["z"])
        ]
        for result in results.values()
    ]
)

GLOBAL_VMIN, GLOBAL_VMAX = np.nanpercentile(
    pooled_values,
    GLOBAL_PERCENTILES,
)

print(
    f"Shared {GLOBAL_PERCENTILES[0]:g}–"
    f"{GLOBAL_PERCENTILES[1]:g} percentile limits:"
)
print(f"  vmin = {GLOBAL_VMIN:.6g}")
print(f"  vmax = {GLOBAL_VMAX:.6g}")
print(f"  pooled finite values = {pooled_values.size:,}")

## Generate and save the heatmaps

In [ ]:
figures = {}

for network in RUNS:
    for component, channel in CHANNELS.items():
        key = (network, component)

        if key not in results:
            continue

        outfile = (
            OUTPUT_DIR
            / f"{network}_rsam_position_time_{channel}.png"
        )

        title = (
            f"{network} nodal {channel} "
            f"{METRIC} amplitude through time"
        )

        fig, ax = plot_heatmap(
            results[key]["plot_data"],
            outfile=outfile,
            title=title,
            metric=METRIC,
            channel=channel,
            cadence=CADENCE,
            use_log10=USE_LOG10,
            vmin=GLOBAL_VMIN,
            vmax=GLOBAL_VMAX,
            figsize=FIGSIZE,
            dpi=DPI,
        )

        figures[key] = (fig, ax)
        plt.show()

## Quality-control summary

In [ ]:
summary_rows = []

for (network, component), result in results.items():
    table = result["table"]

    summary_rows.append(
        {
            "network": network,
            "component": component,
            "channel": result["channel"],
            "seed_ids": len(result["matched_seed_ids"]),
            "original_station_codes": table["x_m_original"].nunique(),
            "merged_station_positions": table["x_m"].nunique(),
            "location_codes": ", ".join(
                sorted(table["location"].unique())
            ),
            "minimum_position_m": table["x_m"].min(),
            "maximum_position_m": table["x_m"].max(),
            "samples": len(table),
            "global_vmin": GLOBAL_VMIN,
            "global_vmax": GLOBAL_VMAX,
        }
    )

summary = (
    pd.DataFrame(summary_rows)
    .sort_values(["network", "component"])
    .reset_index(drop=True)
)

summary

## Optional station-position inventory

This confirms that each station code converts cleanly to the expected
centimetre-based along-line position.

In [ ]:
position_inventory = (
    pd.concat(
        [
            result["table"][
                [
                    "network",
                    "station",
                    "location",
                    "channel",
                    "x_m_original",
                    "x_m",
                ]
            ]
            for result in results.values()
        ],
        ignore_index=True,
    )
    .drop_duplicates()
    .sort_values(
        [
            "network",
            "x_m",
            "location",
            "channel",
        ]
    )
    .reset_index(drop=True)
)

position_inventory

## Position-clustering quality control

Rows with more than one original position show repeated deployment codes
that were merged into one physical receiver location.

In [ ]:
cluster_qc = []

for (network, component), result in results.items():
    clusters = result["position_clusters"].copy()
    clusters.insert(0, "component", component)
    clusters.insert(0, "network", network)
    cluster_qc.append(clusters)

cluster_qc = (
    pd.concat(cluster_qc, ignore_index=True)
    .sort_values(["network", "component", "mean_position_m"])
    .reset_index(drop=True)
)

cluster_qc.loc[
    cluster_qc["original_position_count"] > 1
]